### SCD Type 1 on Product Table

In [0]:
from pyspark.sql.functions import *
#  Source data
df = spark.read.format("delta").table("bankingdata.silver.product")

In [0]:
df.columns

### Hash Column at Source

In [0]:
#  Add HASH column 
df_hash = df.withColumn(
    "src_hash",
    crc32(concat_ws("||",
        col("ProductID").cast("string"),
        col("ProductName"),
        col("ProductType"),
        col("InterestRate").cast("string"),
        col("ModifiedDate").cast("string")
    ))
)

### Target Table

In [0]:
%sql
CREATE TABLE IF NOT EXISTS bankingdata.gold.dim_product
(
    ProductID INT,
    ProductName STRING,
    ProductType STRING,
    InterestRate DOUBLE,
    ModifiedDate TIMESTAMP,
    
    HASHVALUE STRING,
    CREATEDDATE TIMESTAMP,
    CREATEDBY STRING,
    UPDATEDDATE TIMESTAMP,
    UPDATEDBY STRING
)

In [0]:
# Target table
from delta.tables import DeltaTable
table_name = "bankingdata.gold.dim_product"
delta_tgt = DeltaTable.forName(spark, table_name)

### SCD 1 Merge Logic

In [0]:
# MERGE (SCD TYPE 1)
(
    delta_tgt.alias("tgt")
    .merge(
        df_hash.alias("src"),
        "tgt.ProductID = src.ProductID"
    )
    .whenMatchedUpdate(
        condition="tgt.HASHVALUE != src.src_hash",
        set={
            "ProductID": "src.ProductID",
            "ProductName": "src.ProductName",
            "ProductType": "src.ProductType",
            "InterestRate": "src.InterestRate",
            "ModifiedDate": "src.ModifiedDate",
            "HASHVALUE": "src.src_hash",
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("databricks-updated")
        }
    )
    .whenNotMatchedInsert(
        values={
            "ProductID": "src.ProductID",
            "ProductName": "src.ProductName",
            "ProductType": "src.ProductType",
            "InterestRate": "src.InterestRate",
            "ModifiedDate": "src.ModifiedDate",
            "HASHVALUE": "src.src_hash",
            "CREATEDDATE": current_timestamp(),
            "CREATEDBY": lit("databricks"),
            "UPDATEDDATE": current_timestamp(),
            "UPDATEDBY": lit("databricks")
        }
    )
    .execute()
)

In [0]:
%sql
select * from bankingdata.gold.dim_product